# SHIELD — Seven fixed features | Random Forest and XGBoost

Same preprocessing → experiment/checkpoint → results flow as the original master notebook, restricted to seven predefined features. **No IG selection and no imputation.**

Eight protocols:70:30 only,80:20 only,CV5 only,CV10 only,70:30+CV5,70:30+CV10,80:20+CV5,80:20+CV10. Two models =16 results per ratio. DefaultSMOTE0.25; supported ratios0.1,0.25,0.5,0.75and1.0. Change the ratio and rerun from settings for another16 results,stored separately.

Split-only/CV-only use predefined fixed parameters. Split+CV uses the original RF/XGBoost grids,training-only GridSearchCV scoring bankruptcy F1. Ordinary CV-only performs no tuning and is not nested CV. Headline scores for holdout protocols are test results; CV-only scores are pooled held-out fold predictions,not a separate test set.

Accounting availability research identified these seven as calculator candidates from net income,assets,liabilities,equity and retained earnings in complete incorporated-SME accounts. Exact donor formulas and original input transformation are not verified. This notebook trains existing transformed dataset columns,not raw financial statements,and does not deploy a model. Shared/historical test reuse and historical feature selection mean all comparisons remain exploratory.


# PART A — PREPROCESSING
The80:20 example demonstrates training-only SMOTE. Development creates its own partitions from the original seven-column data and never uses the balanced demo arrays as CV input.


## A1. Setup
Configure SMOTE_RATIO to0.1 for the next run;0.25,0.5,0.75and1.0 are also supported. Run the installation/setup cell. If changing versions already loaded in Colab,restart the runtime after installing. Edit Drive paths to match your folders.


In [ ]:
%pip install -q scikit-learn==1.9.0 imbalanced-learn==0.14.2 xgboost==2.0.3 openpyxl matplotlib pandas joblib

import os, random, json, hashlib, platform, time, tempfile, importlib.metadata
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display=print
from sklearn.model_selection import train_test_split,GridSearchCV,StratifiedKFold,ParameterGrid
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,roc_curve,confusion_matrix,classification_report
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from xgboost import XGBClassifier

SMOTE_RATIO=0.25       # Choose0.1,0.25,0.5,0.75or1.0
SEED=42
K_FEATURES=7
SMOTE_K_NEIGHBORS=5
N_JOBS=2              # Search parallelism; each estimator uses one worker
USE_PIPELINE_CACHE=True
NOTEBOOK_VERSION='shield-seven-rf-xgb-eight-protocols-v1'
TARGET_COL='Bankrupt'
LABEL_MAP={0:'Not Bankrupt',1:'Bankrupt'}
SEVEN_FEATURES=["Net Income to Stockholder's Equity", 'Net Income to Total Assets', 'Retained Earnings to Total Assets', 'Debt ratio %', 'Net worth/Assets', 'Equity to Liability', 'Liability to Equity']
assert SMOTE_RATIO in [.1,.25,.5,.75,1.0], 'Choose0.1,0.25,0.5,0.75or1.0'
random.seed(SEED);np.random.seed(SEED)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT=Path('/content/drive/MyDrive/SHIELD_Project')
except ImportError:
    PROJECT=Path.cwd()
    if PROJECT.name=='notebooks':PROJECT=PROJECT.parent
RAW_PATH=str(PROJECT/'data'/'bankruptcy_italian_companies_2023.xlsx')
BASE_REPORT_DIR=str(PROJECT/'reports')
BASE_MODEL_DIR=str(PROJECT/'models_research')
PACKAGE_VERSIONS={name:importlib.metadata.version(name) for name in ['numpy','pandas','scikit-learn','imbalanced-learn','xgboost','joblib']}
print('Seed:',SEED,'SMOTE:',SMOTE_RATIO,'Packages:',PACKAGE_VERSIONS)


## A2. Load Raw Data
*Verification: shape and first rows of the untouched raw dataset.*


In [ ]:
df_raw=pd.read_excel(RAW_PATH) if Path(RAW_PATH).suffix.lower() in ['.xlsx','.xls'] else pd.read_csv(RAW_PATH,float_precision='round_trip')
df_raw.columns=df_raw.columns.str.strip()
RAW_SHA256=hashlib.sha256(Path(RAW_PATH).read_bytes()).hexdigest()
print('Raw shape:',df_raw.shape)
display(df_raw.head())


## A3. Clean and restrict to the seven features
Drop the identifier from model inputs; validate target,unique column names,numeric finite predictors and missingness. Do not remove companies simply because seven ratios coincide. Repeated Company IDs or duplicate full financial records require investigation. No outlier clipping or estimated replacement values.


In [ ]:
assert not df_raw.columns.duplicated().any(), 'Duplicate column names'
assert TARGET_COL in df_raw and set(df_raw[TARGET_COL].dropna().unique())=={0,1}, 'Expected binary Bankrupt target'
assert not df_raw[TARGET_COL].isna().any(), 'Missing target'
assert all(c in df_raw for c in SEVEN_FEATURES), 'Required seven features missing'
if 'Company' in df_raw:
    assert not df_raw.Company.duplicated().any(), 'Repeated Company IDs: investigate grouped splitting'
full_predictors=df_raw.drop(columns=['Company',TARGET_COL],errors='ignore')
assert not full_predictors.duplicated().any(), 'Repeated full financial records: investigate before CV'
df=df_raw[SEVEN_FEATURES+[TARGET_COL]].copy()
for c in SEVEN_FEATURES:df[c]=pd.to_numeric(df[c],errors='raise')
assert not df.isna().any().any(), 'Missing inputs: this experiment does not impute'
assert np.isfinite(df[SEVEN_FEATURES].to_numpy(dtype=float)).all(), 'Nonfinite inputs'
before_shape=df_raw.shape
dropped_cols=[c for c in df_raw if c not in df]
print('Before:',before_shape,'Seven inputs + target:',df.shape)
print('Repeated seven-input combinations (reported,not removed):',df[SEVEN_FEATURES].duplicated().sum())


## A4. Target Column and Labelling
`Bankrupt`: 0 = Not Bankrupt; 1 = Bankrupt.


In [ ]:
y_full = df[TARGET_COL].astype(int)
X_full = df.drop(columns=[TARGET_COL])
label_counts = y_full.value_counts().sort_index()
assert label_counts[1] < label_counts[0], 'This notebook expects Bankrupt (1) to be the minority class.'
assert SMOTE_RATIO > label_counts[1] / label_counts[0], 'SMOTE_RATIO is below the existing minority/majority ratio.'

print('=== TARGET LABELLING VERIFICATION ===')
for cls, count in label_counts.items():
    print(f'  {cls} ({LABEL_MAP[cls]}): {count} rows ({count / len(y_full) * 100:.2f}%)')
print(f'Total rows: {len(y_full)}')
print(f'Features (X) shape: {X_full.shape}')
print(f'Imbalance ratio (Bankrupt:Not Bankrupt) = 1:{label_counts[0] / label_counts[1]:.1f}')

# Hash the actual cleaned data, including feature names, order, dtypes, and values.
data_hasher = hashlib.sha256()
data_hasher.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
data_hasher.update(json.dumps([(str(c), str(df[c].dtype)) for c in df.columns]).encode())
DATA_FINGERPRINT = data_hasher.hexdigest()
print('Cleaned data fingerprint:', DATA_FINGERPRINT[:16])


## A5. Train-Test Split (80:20 demonstration)
*Verification: shapes and stratified class distributions.*


In [ ]:
X_train_demo, X_test_demo, y_train_demo, y_test_demo = train_test_split(
    X_full, y_full, test_size=0.20, random_state=SEED, stratify=y_full)
print('=== SPLIT VERIFICATION (80:20 demo) ===')
print(f'X_train: {X_train_demo.shape}, X_test: {X_test_demo.shape}')
print('y_train distribution:'); print(y_train_demo.value_counts().sort_index())
print('y_test distribution:'); print(y_test_demo.value_counts().sort_index())


## A6. Fixed seven-feature schema — NO IG
The seven remain identical across all models,folds and protocols. Original IG positions are historical references only.

In [ ]:
selected_cols_demo=SEVEN_FEATURES.copy()
X_train_sel_demo=X_train_demo[SEVEN_FEATURES].copy()
X_test_sel_demo=X_test_demo[SEVEN_FEATURES].copy()
feature_reference=pd.DataFrame({'Feature':SEVEN_FEATURES,'Historical_IG_Rank':[5,7,13,14,16,17,20],
    'Historical_IG_Score':[.036753,.035144,.032738,.032479,.032132,.031732,.030660]})
display(feature_reference)
print('Fixed seven inputs;no IG fitted or recalculated.')


## A7. SMOTE â€” Training Set Only (demonstration)
Use the global ratio, not automatic 50:50 balancing. At `0.1`, the resampled minority count is about
one tenth of the majority count. The test set remains untouched.


In [ ]:
print('=== SMOTE VERIFICATION (80:20 demo) ===')
print('SMOTE_RATIO:', SMOTE_RATIO)
print('Train set BEFORE SMOTE:')
print(y_train_demo.value_counts().sort_index())
X_train_bal_demo, y_train_bal_demo = SMOTE(
    sampling_strategy=SMOTE_RATIO, random_state=SEED, k_neighbors=SMOTE_K_NEIGHBORS
).fit_resample(X_train_sel_demo, y_train_demo)
print('\nTrain set AFTER SMOTE:')
print(pd.Series(y_train_bal_demo).value_counts().sort_index())
print(f'\nTrain shape before: {X_train_sel_demo.shape} -> after: {X_train_bal_demo.shape}')
print('\nTest set (UNCHANGED, still imbalanced):')
print(y_test_demo.value_counts().sort_index())
assert len(X_test_sel_demo) == len(X_test_demo)


## A8. Preprocessing Summary


In [ ]:
print('=== PREPROCESSING SUMMARY ===')
print('Raw:',df_raw.shape,'Clean fixed-input table:',df.shape)
print('Target:',LABEL_MAP)
print('Fixed features:',SEVEN_FEATURES)
print('Demo training shapes:',X_train_sel_demo.shape,'→',X_train_bal_demo.shape)
print('Test unchanged:',X_test_sel_demo.shape)
print('PartB uses original X_full/y_full,not balanced demo arrays.')


# PART B — MODEL DEVELOPMENT

Split-only:one train/test fit with fixed parameters.
CV-only:ordinary5/10-fold CV on all original companies with fixed parameters. Each company is held out once;export-only final refit does not generate evaluation scores.
Split+CV:hold out30%or20%,then training-only5/10-fold GridSearchCV. Choose hyperparameters by mean bankruptcy F1,refit the training partition,and evaluate once on its test partition.

No nested CV,no IG,no threshold tuning. A higher test score from a different partition does not establish that partition is intrinsically superior. CV-only and holdout metrics describe different evaluation populations.

Checkpoints save completed experiments and individual CV-only folds. **An interrupted GridSearchCV restarts that particular search**,not the whole notebook. Other completed experiments are reused. SMOTE fits only training subsets.


## B1. Two models,eight protocols and isolated checkpoints

In [ ]:
PROTOCOLS=[
 {'name':'70-30 only','slug':'7030_only','split':.30,'cv':None},
 {'name':'80-20 only','slug':'8020_only','split':.20,'cv':None},
 {'name':'CV=5 only','slug':'cv5_only','split':None,'cv':5},
 {'name':'CV=10 only','slug':'cv10_only','split':None,'cv':10},
 {'name':'70-30 + CV=5','slug':'7030_cv5','split':.30,'cv':5},
 {'name':'70-30 + CV=10','slug':'7030_cv10','split':.30,'cv':10},
 {'name':'80-20 + CV=5','slug':'8020_cv5','split':.20,'cv':5},
 {'name':'80-20 + CV=10','slug':'8020_cv10','split':.20,'cv':10}]
MODELS=['RandomForest','XGBoost']
SCALED_MODELS=[]
MODEL_LABELS={'RandomForest':'Random Forest','XGBoost':'XGBoost'}
SCORE_TYPES={m:'predict_proba' for m in MODELS}
FIXED_PARAMS={
 'RandomForest':{'n_estimators':200,'max_depth':10,'min_samples_split':2},
 'XGBoost':{'n_estimators':200,'max_depth':6,'learning_rate':.1}}
ESTIMATOR_OPTIONS={
 'RandomForest':{'random_state':SEED,'n_jobs':1,'min_samples_leaf':1,'max_features':'sqrt','class_weight':None},
 'XGBoost':{'random_state':SEED,'n_jobs':1,'eval_metric':'logloss'}}
PARAM_GRIDS={
 'RandomForest':{'model__n_estimators':[100,200],'model__max_depth':[6,10,None],'model__min_samples_split':[2,5]},
 'XGBoost':{'model__n_estimators':[100,200],'model__max_depth':[3,6,9],'model__learning_rate':[.01,.1,.2]}}
RATIO_TAG=str(SMOTE_RATIO).replace('.','p')
RUN_SPEC={'version':NOTEBOOK_VERSION,'data_fingerprint':DATA_FINGERPRINT,'raw_sha256':RAW_SHA256,
 'features':SEVEN_FEATURES,'smote_ratio':SMOTE_RATIO,'smote_neighbors':SMOTE_K_NEIGHBORS,'seed':SEED,
 'protocols':PROTOCOLS,'models':MODELS,'fixed':FIXED_PARAMS,'grid':PARAM_GRIDS,
 'estimator_options':ESTIMATOR_OPTIONS,'packages':PACKAGE_VERSIONS,'python':platform.python_version(),
 'threshold':.5,'feature_selection':'fixed7,noIG','scaling':'none',
 'cv_only':'fixedparameter ordinaryCV','split_cv':'training-only GridSearchCV,f1'}
RUN_ID=hashlib.sha256(json.dumps(RUN_SPEC,sort_keys=True).encode()).hexdigest()[:16]
report_dir=f'{BASE_REPORT_DIR}/seven_features_smote_{RATIO_TAG}/{RUN_ID}'
ckpt_dir=f'{report_dir}/checkpoints_master'
model_dir=f'{BASE_MODEL_DIR}/seven_features_smote_{RATIO_TAG}/{RUN_ID}'
for folder in [report_dir,ckpt_dir,model_dir]:os.makedirs(folder,exist_ok=True)
with open(f'{report_dir}/run_config.json','w') as f:json.dump(RUN_SPEC,f,indent=2)
pipeline_memory=joblib.Memory(location=f'{tempfile.gettempdir()}/shield_seven_cache/{RUN_ID}',verbose=0) if USE_PIPELINE_CACHE else None
pd.DataFrame({'Input_Order':range(1,8),'Feature':SEVEN_FEATURES}).to_csv(f'{report_dir}/seven_features.csv',index=False)
for p in PROTOCOLS:
 if p['split'] is not None:
  tr,te=train_test_split(np.arange(len(y_full)),test_size=p['split'],random_state=SEED,stratify=y_full)
  pd.DataFrame({'row_index':np.arange(len(y_full)),'partition':np.where(np.isin(np.arange(len(y_full)),tr),'training','test')}).to_csv(f"{report_dir}/{p['slug']}_partitions.csv",index=False)
print('16 experiments per SMOTE ratio. Full original grids require approximately946 model fits per ratio including CV/refits.')
print('Output:',report_dir)
print('Fixed parameters:',FIXED_PARAMS,'Grids:',PARAM_GRIDS)


## B2. Experiment runner — training-only SMOTE, fixed seven inputs

In [ ]:
def make_estimator(model_name,params=None):
    constructors={'RandomForest':RandomForestClassifier,'XGBoost':XGBClassifier}
    return constructors[model_name](**{**ESTIMATOR_OPTIONS[model_name],**(params or {})})

def make_training_pipeline(model_name,params=None):
    return ImbPipeline([('smote',SMOTE(sampling_strategy=SMOTE_RATIO,random_state=SEED,k_neighbors=SMOTE_K_NEIGHBORS)),
        ('model',make_estimator(model_name,params))],memory=pipeline_memory)

def make_cv(folds):
    return StratifiedKFold(n_splits=folds, shuffle=True, random_state=SEED)

def make_grid(model_name, folds):
    return GridSearchCV(make_training_pipeline(model_name), PARAM_GRIDS[model_name],
                        cv=make_cv(folds), scoring='f1', n_jobs=N_JOBS,
                        refit=True, error_score='raise')

def check_resampling_feasible(y, context):
    counts = pd.Series(y).value_counts()
    if set(counts.index) != {0, 1}:
        raise ValueError(f'{context}: both target classes must be present.')
    desired_minority = int(SMOTE_RATIO * int(counts[0]))
    if desired_minority <= counts[1]:
        raise ValueError(f'{context}: SMOTE_RATIO={SMOTE_RATIO} would not add minority samples; '
                         'increase the ratio for this dataset/fold.')
    if counts[1] <= SMOTE_K_NEIGHBORS:
        raise ValueError(f'{context}: too few bankrupt samples for SMOTE k_neighbors={SMOTE_K_NEIGHBORS}.')

def check_cv_feasible(X, y, folds, context):
    counts = pd.Series(y).value_counts()
    if counts.min() < folds:
        raise ValueError(f'{context}: fewer minority samples than CV folds.')
    for fold_id, (train_idx, _) in enumerate(make_cv(folds).split(X, y), start=1):
        check_resampling_feasible(y.iloc[train_idx], f'{context}, training fold {fold_id}')

def compute_metrics(y_true, y_pred, y_score):
    return {
        'Accuracy (%)': accuracy_score(y_true, y_pred) * 100,
        'Precision (%)': precision_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'Recall (%)': recall_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'F1-Score (%)': f1_score(y_true, y_pred, pos_label=1, zero_division=0) * 100,
        'ROC-AUC (%)': roc_auc_score(y_true, y_score) * 100,
    }

def predict_outputs(fitted_pipeline,X):
    classes=fitted_pipeline.named_steps['model'].classes_.tolist()
    assert classes==[0,1]
    score=np.asarray(fitted_pipeline.predict_proba(X)[:,1],dtype=float)
    return (score>=.5).astype(int),score

def selected_features(fitted_pipeline):
    columns=fitted_pipeline.feature_names_in_.tolist()
    assert columns==SEVEN_FEATURES,'Checkpoint/model feature order mismatch'
    return columns

def checkpoint_paths(slug, model_name):
    base = f'{ckpt_dir}/{slug}_{model_name}'
    return {'result': f'{base}_result.json', 'preds': f'{base}_preds.npz',
            'model': f'{base}_model.joblib'}

def load_checkpoint(slug, model_name):
    paths = checkpoint_paths(slug, model_name)
    if not all(os.path.exists(p) for p in paths.values()):
        return None
    with open(paths['result']) as f:
        result = json.load(f)
    if result.get('Run_ID') != RUN_ID:
        return None
    with np.load(paths['preds'], allow_pickle=False) as arrays:
        return result, arrays['y_true'].copy(), arrays['y_pred'].copy(), arrays['y_score'].copy()

def save_checkpoint(slug, model_name, result, y_true, y_pred, y_score, fitted_model):
    paths = checkpoint_paths(slug, model_name)
    # Save the result last: it marks an experiment as complete.
    joblib.dump(fitted_model, paths['model'])
    np.savez_compressed(paths['preds'], y_true=np.asarray(y_true),
                        y_pred=np.asarray(y_pred), y_score=np.asarray(y_score))
    with open(paths['result'], 'w') as f:
        json.dump(result, f, indent=2)

def run_ordinary_cv(model_name, config):
    folds = config['cv']
    check_cv_feasible(X_full, y_full, folds, 'Ordinary fixed-parameter CV')
    y_pred = np.empty(len(y_full), dtype=int)
    y_score = np.empty(len(y_full), dtype=float)
    visited = np.zeros(len(y_full), dtype=bool)
    fold_records = []
    for fold_id, (train_idx, val_idx) in enumerate(make_cv(folds).split(X_full, y_full), start=1):
        fold_base = f"{ckpt_dir}/{config['slug']}_{model_name}_fold_{fold_id}"
        fold_json, fold_npz = f'{fold_base}.json', f'{fold_base}.npz'
        cached = None
        if os.path.exists(fold_json) and os.path.exists(fold_npz):
            with open(fold_json) as f:
                record = json.load(f)
            if record.get('Run_ID') == RUN_ID:
                with np.load(fold_npz, allow_pickle=False) as arrays:
                    if np.array_equal(arrays['val_idx'], val_idx):
                        cached = arrays['y_pred'].copy(), arrays['y_score'].copy(), record
        if cached is not None:
            fold_pred, fold_score, record = cached
            print(f'  [FOLD CHECKPOINT] {model_name} | {config["name"]} | fold {fold_id}/{folds}')
        else:
            fold_start = time.perf_counter()
            print(f'  [ORDINARY CV] {model_name} | {config["name"]} | fold {fold_id}/{folds}')
            fitted_fold = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
            fitted_fold.fit(X_full.iloc[train_idx], y_full.iloc[train_idx])
            fold_pred, fold_score = predict_outputs(fitted_fold, X_full.iloc[val_idx])
            record = {'Run_ID': RUN_ID, 'Fold': fold_id,
                      'Configured_Params': FIXED_PARAMS[model_name],
                      'Parameter_Source': 'Predefined fixed parameters; no tuning',
                      'Score_Type': SCORE_TYPES[model_name],
                      'Selected_Features': selected_features(fitted_fold),
                      'Train_Count': len(train_idx), 'Validation_Count': len(val_idx),
                      'Elapsed_Seconds': time.perf_counter() - fold_start,
                      **compute_metrics(y_full.iloc[val_idx], fold_pred, fold_score)}
            np.savez_compressed(fold_npz,val_idx=val_idx,train_idx=train_idx,y_true=y_full.iloc[val_idx],y_pred=fold_pred,y_score=fold_score)
            with open(fold_json, 'w') as f:
                json.dump(record, f, indent=2)
        assert not visited[val_idx].any(), 'A held-out CV row was predicted twice.'
        y_pred[val_idx], y_score[val_idx], visited[val_idx] = fold_pred, fold_score, True
        fold_records.append(record)
    assert visited.all(), 'Every row must receive exactly one held-out CV prediction.'
    pd.DataFrame(fold_records).to_csv(
        f"{report_dir}/{config['slug']}_{model_name}_fold_results.csv", index=False)

    # One full-data fit with the SAME fixed settings, for export only (no grid search).
    print(f'  [FIXED FINAL REFIT] {model_name} | {config["name"]} | all cleaned data')
    check_resampling_feasible(y_full, 'Final fixed full-data refit')
    fitted_model = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
    fitted_model.fit(X_full, y_full)
    return y_pred, y_score, fold_records, fitted_model

def run_experiment(model_name, config):
    started = time.perf_counter()
    split_size, cv_folds = config['split'], config['cv']
    result = {'Model': model_name, 'Experiment': config['name'], 'Split': split_size, 'CV': cv_folds,
              'SMOTE_Ratio': float(SMOTE_RATIO), 'Run_ID': RUN_ID,
              'Score_Type': SCORE_TYPES[model_name],
              'Scaling': 'StandardScaler before SMOTE' if model_name in SCALED_MODELS else 'None'}

    if split_size is None:
        y_pred, y_score, fold_records, fitted_model = run_ordinary_cv(model_name, config)
        y_true = y_full.to_numpy()
        fold_scores = [r['F1-Score (%)'] for r in fold_records]
        result.update(compute_metrics(y_true, y_pred, y_score))
        result.update({'Best_Params': FIXED_PARAMS[model_name],
                       'CV_F1_Mean (%)': float(np.mean(fold_scores)),
                       'CV_F1_Std (%)': float(np.std(fold_scores, ddof=1)),
                       'CV_ROC_AUC_Mean (%)': float(np.mean([r['ROC-AUC (%)'] for r in fold_records])),
                       'Parameter_Source': 'Predefined fixed parameters; no tuning',
                       'Evaluation': 'Ordinary CV held-out predictions (fixed parameters)'})
    else:
        X_train, X_test, y_train, y_test = train_test_split(
            X_full, y_full, test_size=split_size, random_state=SEED, stratify=y_full)
        if cv_folds is None:
            check_resampling_feasible(y_train, 'Holdout training set')
            fitted_model = make_training_pipeline(model_name, FIXED_PARAMS[model_name])
            fitted_model.fit(X_train, y_train)
            result['Best_Params'] = FIXED_PARAMS[model_name]
            result['CV_F1_Mean (%)'] = None
            result['Parameter_Source'] = 'Predefined fixed parameters; no tuning'
        else:
            check_cv_feasible(X_train, y_train, cv_folds, 'Holdout training CV')
            grid = make_grid(model_name, cv_folds)
            grid.fit(X_train, y_train)
            fitted_model = grid.best_estimator_
            result['Best_Params'] = grid.best_params_
            result['CV_F1_Mean (%)'] = grid.best_score_ * 100
            result['Parameter_Source'] = 'GridSearchCV on training partition only'
            pd.DataFrame(grid.cv_results_).to_csv(
                f"{report_dir}/{config['slug']}_{model_name}_grid_results.csv", index=False)
        y_pred, y_score = predict_outputs(fitted_model, X_test)
        y_true = y_test.to_numpy()
        result.update(compute_metrics(y_true, y_pred, y_score))
        result['Evaluation'] = 'Held-out test partition; excluded from this fit/search'

    result['Selected_Features'] = selected_features(fitted_model)
    result['Evaluation_Count'] = len(y_true)
    result['Bankrupt_Support'] = int(np.sum(y_true == 1))
    result['Elapsed_Seconds'] = time.perf_counter() - started
    return result, y_true, y_pred, y_score, fitted_model

print('Runner:fixed7→training-onlySMOTE→model;noIG. CV-only=fixedparameter;split+CV=training-onlytuning.')


## B3. Run16 experiments
Resume completed checkpoints. Search results include selected parameters and full candidate CV scores. Model checkpoints are research artifacts and do not replace production files.

In [ ]:
all_results = []
predictions = {cfg['slug']: {} for cfg in PROTOCOLS}
execution_order = [(cfg, model_name) for cfg in PROTOCOLS for model_name in MODELS]

for cfg, model_name in execution_order:
    ckpt = load_checkpoint(cfg['slug'], model_name)
    if ckpt is not None:
        print(f"[CHECKPOINT] Skipping {model_name} | {cfg['name']} (already done)")
        result, y_true, y_pred, y_score = ckpt
    else:
        print(f"[RUNNING] {model_name} | {cfg['name']} ...")
        result, y_true, y_pred, y_score, fitted_model = run_experiment(model_name, cfg)
        save_checkpoint(cfg['slug'], model_name, result, y_true, y_pred, y_score, fitted_model)
    print(f"  Accuracy: {result['Accuracy (%)']:.2f}% | Precision: {result['Precision (%)']:.2f}% | "
          f"Recall: {result['Recall (%)']:.2f}% | F1: {result['F1-Score (%)']:.2f}% | "
          f"ROC-AUC: {result['ROC-AUC (%)']:.2f}%")
    all_results.append(result)
    predictions[cfg['slug']][model_name] = (y_true, y_pred, y_score)

protocol_order = {cfg['name']: i for i, cfg in enumerate(PROTOCOLS)}
model_order = {name: i for i, name in enumerate(MODELS)}
results_df = pd.DataFrame(all_results)
results_df['_protocol_order'] = results_df['Experiment'].map(protocol_order)
results_df['_model_order'] = results_df['Model'].map(model_order)
results_df = results_df.sort_values(['_protocol_order', '_model_order']).drop(
    columns=['_protocol_order', '_model_order']).reset_index(drop=True)
# Preserve the original headline column order; traceability columns follow it.
original_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                    'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'Best_Params', 'CV_F1_Mean (%)']
results_df = results_df[original_columns + [c for c in results_df if c not in original_columns]]
expected_count = len(PROTOCOLS) * len(MODELS)
assert len(results_df) == expected_count
print(f'\n=== ALL {expected_count} EXPERIMENTS COMPLETE ===')
display(results_df[original_columns])


## B4. Full Metrics Table and Classification Reports
*The same report layout and target labels as the original.*


In [ ]:
for cfg in PROTOCOLS:
    for model_name in MODELS:
        y_true, y_pred, y_score = predictions[cfg['slug']][model_name]
        print(f"\n=== Classification Report: {model_name} | {cfg['name']} ===")
        print(classification_report(y_true, y_pred, labels=[0, 1],
              target_names=['Not Bankrupt', 'Bankrupt'], zero_division=0))


## B5. Comparison bars,confusion matrices and ROC curves
Two models per protocol; both PNG and SVG exports. Curves use actual held-out prediction scores. No fabricated ROC curves.

In [ ]:
MODEL_COLORS={'RandomForest':'#4C93E0','XGBoost':'#D866F5'}
MODEL_CMAPS={'RandomForest':'Blues','XGBoost':'Purples'}
metrics_labels = ['Accuracy (%)', 'ROC-AUC (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)']
for cfg in PROTOCOLS:
    slug, pname = cfg['slug'], cfg['name']
    eval_suffix = ' [Ordinary CV held-out]' if cfg['split'] is None else ''
    rows = {name: results_df[(results_df['Experiment'] == pname) &
                            (results_df['Model'] == name)].iloc[0] for name in MODELS}

    fig, ax = plt.subplots(figsize=(14, 6))
    x = np.arange(len(metrics_labels))
    width = 0.8 / len(MODELS)
    for model_index, model_name in enumerate(MODELS):
        values = [rows[model_name][metric] for metric in metrics_labels]
        offset = (model_index - (len(MODELS) - 1) / 2) * width
        bars = ax.bar(x + offset, values, width, label=MODEL_LABELS[model_name],
                      color=MODEL_COLORS[model_name])
        for bar in bars:
            ax.annotate(f'{bar.get_height():.1f}%', (bar.get_x()+bar.get_width()/2, bar.get_height()),
                        textcoords='offset points', xytext=(0,3), ha='center', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(metrics_labels); ax.set_ylabel('Score (%)'); ax.set_ylim(0,110)
    ax.set_title(f'Model Performance Comparison â€” {pname}{eval_suffix} (Target: Bankrupt class)', weight='bold')
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_bar_comparison.png', dpi=300); plt.savefig(f'{report_dir}/{slug}_bar_comparison.svg',bbox_inches='tight'); plt.show(); plt.close()

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, model_name in zip(axes.flat, MODELS):
        y_true, y_pred, _ = predictions[slug][model_name]
        cm = confusion_matrix(y_true, y_pred, labels=[0,1])
        ax.imshow(cm, cmap=MODEL_CMAPS[model_name])
        ax.set_title(f'{MODEL_LABELS[model_name]} Confusion Matrix', weight='bold')
        ax.set_xticks([0,1]); ax.set_xticklabels(['Not Bankrupt (0)', 'Bankrupt (1)'])
        ax.set_yticks([0,1]); ax.set_yticklabels(['Not Bankrupt (0)', 'Bankrupt (1)'])
        ax.set_xlabel('Predicted Label'); ax.set_ylabel('True Label')
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                        color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=14)
    plt.suptitle(f'Confusion Matrices â€” {pname}{eval_suffix}', weight='bold'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_confusion_matrices.png', dpi=300); plt.savefig(f'{report_dir}/{slug}_confusion_matrices.svg',bbox_inches='tight'); plt.show(); plt.close()

    fig, ax = plt.subplots(figsize=(8, 6))
    for model_name in MODELS:
        y_true, _, y_score = predictions[slug][model_name]
        fpr, tpr, _ = roc_curve(y_true, y_score)
        ax.plot(fpr, tpr, color=MODEL_COLORS[model_name],
                label=f"{MODEL_LABELS[model_name]} (AUC = {rows[model_name]['ROC-AUC (%)']/100:.4f})")
    ax.plot([0,1], [0,1], linestyle='--', color='gray')
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Receiver Operating Characteristic (ROC) Curve â€” {pname}{eval_suffix}', weight='bold')
    ax.legend(loc='lower right'); plt.tight_layout()
    plt.savefig(f'{report_dir}/{slug}_roc_curve.png', dpi=300); plt.savefig(f'{report_dir}/{slug}_roc_curve.svg',bbox_inches='tight'); plt.show(); plt.close()
    print(f'Saved 3 charts for: {pname}')


## B6. Save Final Result Tables
The headline slide table keeps the original column layout. Full results retain traceability and fold details.


In [ ]:
results_df.to_csv(f'{report_dir}/master_results_full.csv', index=False)
slide_columns = ['Model', 'Experiment', 'Split', 'CV', 'Accuracy (%)', 'Precision (%)',
                 'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)', 'CV_F1_Mean (%)']
slide_df = results_df[slide_columns]
slide_df.to_csv(f'{report_dir}/master_results_slide.csv', index=False)
with open(f'{report_dir}/master_results_full.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print('=== FINAL RESULTS TABLE ===')
display(slide_df)
print(f'\nAll tables, charts ({3 * len(PROTOCOLS)} images), and model checkpoints ({len(PROTOCOLS) * len(MODELS)} .joblib files) saved to:')
print(f'  {report_dir}')
print('\nReview CV and held-out results before selecting a final model. The previous winner is a candidate, not an assumed new winner.')


## B7. Research model and fixed seven-input schema
Each completed experiment already has a fitted research pipeline checkpoint. This cell records the schema and selected RF80:20+CV10 reference;it does not automatically choose a winner or deploy it.

In [ ]:
reference=load_checkpoint('8020_cv10','RandomForest')
assert reference is not None,'Complete the experiment first'
manifest={'Run_ID':RUN_ID,'Features':SEVEN_FEATURES,'Input_Count':7,
 'Expected_Input':'Existing dataset feature representation;raw accounting transformation not verified',
 'No_IG':True,'No_Imputation':True,'Threshold':.5,'SMOTE_Ratio':SMOTE_RATIO,
 'Research_Reference_Only':reference[0],'Production_Replaced':False}
with open(f'{model_dir}/seven_input_manifest.json','w') as f:json.dump(manifest,f,indent=2)
print('Research pipelines:',ckpt_dir)
print('Schema manifest:',model_dir)
print('No production model replaced.')


## Interpretation and limitations
-16results per ratio;32across0.25and1.0. No50:50,noLR/SVM,noIG. HistoricalIG ranks are references only.
-CV-only is fixed-parameter ordinaryCV,no full-dataset tuning before score reporting. Split+CV uses training-only tuning;its bestCVF1 is a selection score,not an independent test result.
-Threshold0.5 and preprocessing are predefined. No additional scaling introduced;the seven input representation matches the controlled prior experiments.
-The seven-feature fixed models use200trees for bothRF/XGBoost. Prior focused XGBoost comparisons used100in some experiments;do not confuse those configurations.
-Checkpoints protect completed experiments and CV-only folds. Interrupted grid searches restart that search. Full original grids can be slow;keep runtime expectations separate from16result rows.
-Scores from reused historical partitions are exploratory. OrdinaryCV pooled scores and different holdout ratios are not interchangeable evidence that one data split is inherently best. New independent validation remains needed.
-Accounting availability,exact donor feature definition,and correct transformation are separate checks. This experiment does not make the seven automatically available or model-ready for allSMEs.
-[Leakage guidance](https://imbalanced-learn.org/stable/common_pitfalls.html) · [GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
